In [ ]:

from pathlib import Path
import json

import pandas as pd

def find_project_root(start: Path) -> Path:
    """Walk up from the current folder until we find the repo root."""
    for candidate in (start, *start.parents):
        if (candidate / "data" / "input").is_dir():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "structured"
TABLES_DIR = OUTPUT_DIR / "tables"
JSON_PATH = OUTPUT_DIR / "document_structure.json"
REPORT_PATH = OUTPUT_DIR / "validation_report.json"


def validate_table(table, table_number):
    """Check table structure and flag cases that deserve inspection."""
    cells = table.get("data", {}).get("table_cells", [])
    issues = []
    occupied_positions = {}

    if not cells:
        issues.append("No table cells found")
        return {
            "table_number": table_number,
            "page": None,
            "cell_count": 0,
            "issues": issues,
            "needs_review": True,
        }

    # Track each cell's occupied grid positions, including spans.
    for cell in cells:
        row_start = cell.get("start_row_offset_idx", 0)
        col_start = cell.get("start_col_offset_idx", 0)
        row_span = cell.get("row_span", 1)
        col_span = cell.get("col_span", 1)
        cell_text = cell.get("text", "")

        if row_span < 1 or col_span < 1:
            issues.append(f"Invalid span for cell: {cell_text!r}")
            continue

        if row_span > 1 or col_span > 1:
            # A merged cell is not automatically an error.
            # We flag it because a flat CSV may lose layout information.
            issues.append(
                f"Merged cell: {cell_text!r}, "
                f"row_span={row_span}, col_span={col_span}"
            )

        for row in range(row_start, row_start + row_span):
            for col in range(col_start, col_start + col_span):
                position = (row, col)

                if position in occupied_positions:
                    previous_text = occupied_positions[position]
                    issues.append(
                        f"Overlapping cells at {position}: "
                        f"{previous_text!r} and {cell_text!r}"
                    )
                else:
                    occupied_positions[position] = cell_text

    # Read the source page number, if available.
    provenance = table.get("prov", [])
    page_number = provenance[0].get("page_no") if provenance else None

    # Check the existing CSV, if one exists.
    csv_path = TABLES_DIR / f"table_{table_number:03}.csv"
    csv_exists = csv_path.exists()
    csv_shape = None
    csv_duplicate_rows = None

    if csv_exists:
        try:
            dataframe = pd.read_csv(csv_path, dtype=str).fillna("")
            csv_shape = list(dataframe.shape)
            csv_duplicate_rows = int(dataframe.duplicated().sum())
        except Exception as error:
            issues.append(f"Could not read CSV: {error}")
    else:
        issues.append("Default CSV export missing")

    # Merged cells are review flags, not proof of incorrect extraction.
    needs_review = bool(issues)

    return {
        "table_number": table_number,
        "page": page_number,
        "cell_count": len(cells),
        "csv_exists": csv_exists,
        "csv_shape": csv_shape,
        "csv_duplicate_rows": csv_duplicate_rows,
        "issues": issues,
        "needs_review": needs_review,
    }


def main():
    if not JSON_PATH.exists():
        raise FileNotFoundError(f"Structured JSON not found: {JSON_PATH}")

    data = json.loads(JSON_PATH.read_text(encoding="utf-8"))
    tables = data.get("tables", [])

    results = [
        validate_table(table, number)
        for number, table in enumerate(tables, start=1)
    ]

    report = {
        "source_json": str(JSON_PATH.relative_to(PROJECT_ROOT)),
        "table_count": len(results),
        "tables_needing_review": sum(
            result["needs_review"] for result in results
        ),
        "tables": results,
    }

    REPORT_PATH.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    print(f"Tables inspected: {len(results)}")
    print(f"Tables flagged for review: {report['tables_needing_review']}")

    for result in results:
        print(
            f"\nTable {result['table_number']} | "
            f"Page {result['page']} | "
            f"Cells: {result['cell_count']}"
        )

        for issue in result["issues"][:5]:
            print(f"  - {issue}")

        if len(result["issues"]) > 5:
            print(f"  ... and {len(result['issues']) - 5} more issues")

    print(f"\nValidation report saved to: {REPORT_PATH}")


if __name__ == "__main__":
    main()


Tables inspected: 12
Tables flagged for review: 3

Table 1 | Page 1 | Cells: 4

Table 2 | Page 1 | Cells: 69
  - Merged cell: 'Company Data', row_span=1, col_span=4
  - Merged cell: 'Market Cap (cr)', row_span=1, col_span=3
  - Merged cell: 'Enterprise Value (cr)', row_span=1, col_span=3
  - Merged cell: 'Outstanding Shares (cr)', row_span=1, col_span=3
  - Merged cell: 'Free Float', row_span=1, col_span=3
  ... and 6 more issues

Table 3 | Page 1 | Cells: 56

Table 4 | Page 2 | Cells: 113

Table 5 | Page 2 | Cells: 45
  - Merged cell: 'Old estimates', row_span=1, col_span=2
  - Merged cell: 'New estimates', row_span=1, col_span=2
  - Merged cell: 'Change %', row_span=1, col_span=2

Table 6 | Page 2 | Cells: 82
  - Merged cell: '10% holding discount to investments', row_span=1, col_span=2

Table 7 | Page 3 | Cells: 132

Table 8 | Page 3 | Cells: 96

Table 9 | Page 3 | Cells: 138

Table 10 | Page 3 | Cells: 118

Table 11 | Page 4 | Cells: 27

Table 12 | Page 4 | Cells: 21

Validation re